In [ ]:
!pip install umap-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import umap  # Thư viện UMAP
import time

# 1. Tải dữ liệu Digits (8x8 images of digits)
print("Đang tải dữ liệu Digits...")
digits = load_digits()
X, y = digits.data, digits.target
X_scaled = StandardScaler().fit_transform(X)

# 2. Cấu hình các bộ giảm chiều
# t-SNE tập trung vào cấu trúc địa phương (local structure)
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=-1)

# UMAP cân bằng giữa cấu trúc địa phương và toàn cục (global structure)
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)

# 3. Thực thi t-SNE
print("Đang chạy t-SNE...")
start_tsne = time.time()
X_tsne = tsne.fit_transform(X_scaled)
time_tsne = time.time() - start_tsne

# 4. Thực thi UMAP
print("Đang chạy UMAP...")
start_umap = time.time()
X_umap = reducer.fit_transform(X_scaled)
time_umap = time.time() - start_umap

# 5. Vẽ biểu đồ so sánh
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('So sánh Giảm chiều dữ liệu: t-SNE vs UMAP (Digits Dataset)', fontsize=16, fontweight='bold')

# Plot t-SNE
scatter1 = ax1.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='Spectral', s=15, alpha=0.8)
ax1.set_title(f't-SNE (Thời gian: {time_tsne:.2f}s)', fontsize=12)
ax1.set_xlabel('t-SNE 1')
ax1.set_ylabel('t-SNE 2')
plt.colorbar(scatter1, ax=ax1, label='Digit Value')

# Plot UMAP
scatter2 = ax2.scatter(X_umap[:, 0], X_umap[:, 1], c=y, cmap='Spectral', s=15, alpha=0.8)
ax2.set_title(f'UMAP (Thời gian: {time_umap:.2f}s)', fontsize=12)
ax2.set_xlabel('UMAP 1')
ax2.set_ylabel('UMAP 2')
plt.colorbar(scatter2, ax=ax2, label='Digit Value')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print(f"\nKết quả:")
print(f"- Thời gian t-SNE: {time_tsne:.4f} giây")
print(f"- Thời gian UMAP: {time_umap:.4f} giây")

# Tại sao UMAP lại được ưa chuộng hơn?

## Tính bảo toàn cấu trúc: Nếu bạn nhìn vào biểu đồ UMAP, các cụm số có đặc điểm giống nhau (ví dụ số 1 và số 7) thường nằm gần nhau hơn so với t-SNE. t-SNE chỉ quan tâm đến việc gom nhóm các điểm giống hệt nhau, còn UMAP cố gắng giữ đúng khoảng cách tương đối giữa các nhóm lớn.

## Tốc độ (Speed): Với tập dữ liệu lớn (hàng trăm ngàn điểm), UMAP sẽ bỏ xa t-SNE về mặt hiệu năng.

## Khả năng mở rộng: UMAP cho phép bạn "train" trên một tập dữ liệu và "transform" dữ liệu mới vào không gian đã giảm chiều đó (giống như PCA), điều mà t-SNE nguyên bản không làm được một cách dễ dàng.